In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_range_coarse_S5_20261103_180000.csv"; None = newest run
# Run to compare against (overlay below the overview), e.g. the same filter before the PLL supply moved to the
# E36312A: LOOP_FILTER _161640, DEFAULT_FILTER _160525 (2026-10-13). None = no comparison
COMPARE_CSV = None  # e.g. PLL_DIR / "pll_range_S5_20261013_161640.csv"

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_range_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")


def load_runs(csv):
    """One row per reference frequency of a range sweep, with:
    good      lock bit set, carrier on the pad at f_vco / N (divider check) and the lock bit still set after the
              measurement; for a lock check only (status "lock_only", range_sweep measure=False): the lock bit set at
              both reads;
    measured  carrier measured at the right frequency (the curves and jitter exist), also when the lock was lost
              during the measurement;
    category  locked / lost lock during the measurement / locked or lost lock in a lock check only / lock bit set,
              check failed (no carrier, wrong frequency, error) / not locked / no VCO setting;
    filter    the loop filter of the point (Icp C1 C2 R1 codes), which may differ per point (coarse sweep)."""
    df = pd.read_csv(csv).sort_values("f_ref").reset_index(drop=True)
    lock_only = df["status"] == "lock_only"
    df["measured"] = (df["status"] == "ok") & (df["divider_ok"] == 1)
    df["good"] = (df["measured"] | lock_only) & (df["locked_end"] == 1)
    df["category"] = np.select(
        [df["status"] == "no_vco_setting", df["status"] == "not_locked", lock_only & df["good"], lock_only,
         df["good"], df["measured"]],
        ["no VCO setting", "not locked", "locked (lock check only)", "lost lock (lock check only)", "locked",
         "lost lock during the measurement"], "lock bit set, check failed")
    df["filter"] = [f"Icp {r.set_current:.0f} C1 {r.set_c1:.0f} C2 {r.set_c2:.0f} R1 {r.set_r1:.0f}"
                    for r in df.itertuples()]
    return df


def filter_txt(df):
    """The loop filter of a run, or "per point" when it changes over the sweep."""
    labels = df["filter"].unique()
    return labels[0] if len(labels) == 1 else "per point"


runs = load_runs(RESULT_CSV)
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=["f_ref", "kind"])
FILTER_TXT = filter_txt(runs)

# Lock range: the longest stretch of neighbouring locked points; each edge lies between the last locked point and the
# next measured one (the sweep step).
good = np.r_[runs["good"].to_numpy(), False]
best, start = (0, 0), None
for i, ok in enumerate(good):
    if ok and start is None:
        start = i
    if not ok and start is not None:
        best = max(best, (start, i), key=lambda b: b[1] - b[0])
        start = None
if best[1] > best[0]:
    LOCK = (runs["f_ref"][best[0]], runs["f_ref"][best[1] - 1])
    below = runs["f_ref"][best[0] - 1] if best[0] > 0 else None
    above = runs["f_ref"][best[1]] if best[1] < len(runs) else None
    print(f"{RESULT_CSV.name}, loop filter {FILTER_TXT}: locks from ref {LOCK[0] / 1e6:.3f} to {LOCK[1] / 1e6:.3f} MHz "
          f"(VCO {LOCK[0] * 128 / 1e9:.3f} - {LOCK[1] * 128 / 1e9:.3f} GHz), {best[1] - best[0]} points")
    print("  lower edge between " + (f"{below / 1e6:.3f} and " if below else "below the sweep, at or under ")
          + f"{LOCK[0] / 1e6:.3f} MHz; upper edge between {LOCK[1] / 1e6:.3f}"
          + (f" and {above / 1e6:.3f} MHz" if above else " MHz and above the sweep"))
else:
    LOCK = None
    print(f"{RESULT_CSV.name}, loop filter {FILTER_TXT}: no locked point")
print(runs["category"].value_counts().to_string())

# Jitter-power figure of merit: 10 log10((jitter / 1 s)^2 x P / 1 mW), full-curve jitter over the common band
fom = 10 * np.log10((runs["time_jitter_trace_common_ps"] * 1e-12) ** 2 * runs["pll_p"] * 1e3)
display(pd.DataFrame({
    "ref [MHz]": runs["f_ref"] / 1e6,
    "VCO [MHz]": runs["f_vco"] / 1e6,
    "filter": runs["filter"],
    "|Kvco| request": runs["kvco_request"] if "kvco_request" in runs else "-",
    "setting": [f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}"
                if pd.notna(r.vco_tune_coarse) else "-" for r in runs.itertuples()],
    "VCO pick": runs["vco_pick"] if "vco_pick" in runs else "lut",  # "fastest": no table setting
    "Vctrl pred [V]": runs["vctrl_pred"],
    "|Kvco| pred [MHz/V]": runs["kvco_pred"] / 1e6,
    "result": runs["category"],
    "status": runs["status"],
    "VCO meas [MHz]": runs["f_vco_meas"] / 1e6,
    "carrier [dBm]": runs["carrier_dbm"],
    "P [mW]": runs["pll_p"] * 1e3,
    "jitter curve [ps]": runs["time_jitter_trace_ps"],
    "jitter common band [ps]": runs["time_jitter_trace_common_ps"],
    "jitter noise [ps]": runs["time_jitter_noise_ps"],
    "jitter spurs [ps]": runs["time_jitter_spur_ps"],
    "FOM [dB]": fom,
}).round(3))

pll_range_coarse_S5_20261103_174312.csv, loop filter per point: locks from ref 1.381 to 13.200 MHz (VCO 0.177 - 1.690 GHz), 8 points
  lower edge between 1.000 and 1.381 MHz; upper edge between 13.200 and 18.223 MHz
category
locked                              11
lost lock during the measurement     2
not locked                           2


,ref [MHz],VCO [MHz],filter,|Kvco| request,setting,VCO pick,Vctrl pred [V],|Kvco| pred [MHz/V],result,status,VCO meas [MHz],carrier [dBm],P [mW],jitter curve [ps],jitter common band [ps],jitter noise [ps],jitter spurs [ps],FOM [dB]
0,1.000,128.000,Icp 0 C1 7 C2 3 R1 3,min,c13 min 14 max 15,lut,0.653,278.770,lost lock during the measurement,ok,128.000,0.783,0.185,1074.926,1074.825,1073.965,45.443,-186.696
1,1.381,176.718,Icp 0 C1 7 C2 3 R1 3,min,c6 min 14 max 15,lut,0.655,402.542,locked,ok,176.717,-0.006,0.191,774.722,774.527,774.363,23.593,-189.401
2,1.906,243.977,Icp 0 C1 7 C2 3 R1 3,2000000000.0,c7 min 14 max 15,lut,0.593,2000.235,locked,ok,243.977,1.332,0.204,355.794,355.563,354.042,35.256,-195.892
3,2.632,336.836,Icp 0 C1 3 C2 3 R1 3,2000000000.0,c14 min 13 max 6,lut,0.638,1993.291,locked,ok,336.836,0.842,0.247,373.856,373.630,372.842,27.525,-194.628
4,3.633,465.038,Icp 0 C1 3 C2 3 R1 3,2000000000.0,c14 min 11 max 2,lut,0.646,1959.268,locked,ok,465.038,1.211,0.291,381.107,380.982,380.829,14.555,-193.749
5,5.016,642.035,Icp 3 C1 3 C2 3 R1 3,2000000000.0,c12 min 9 max 0,lut,0.642,1990.358,locked,ok,642.034,1.186,0.378,126.422,126.069,125.940,11.033,-202.214
6,6.925,886.397,Icp 3 C1 3 C2 3 R1 3,2000000000.0,c4 min 11 max 14,lut,0.611,2011.954,locked,ok,886.396,1.144,0.360,119.364,118.793,118.875,10.793,-202.945
7,9.561,1223.765,Icp 7 C1 0 C2 0 R1 7,2000000000.0,c3 min 9 max 14,lut,0.619,1992.897,locked,ok,1223.764,1.357,0.464,87.974,86.752,87.140,12.080,-204.572
8,13.200,1689.537,Icp 7 C1 0 C2 0 R1 7,2000000000.0,c1 min 9 max 15,lut,0.559,2297.861,locked,ok,1689.536,0.831,0.505,524.119,523.826,523.920,14.422,-188.583
9,18.223,2332.584,Icp 7 C1 0 C2 0 R1 7,2000000000.0,c0 min 8 max 15,lut,0.534,2543.530,not locked,not_locked,NaN,NaN,0.573,NaN,NaN,NaN,NaN,NaN


In [18]:
import sys

sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots
from sw.lib.pll_util import bin_phase_noise

INK, GRID = "#52514e", "#e4e3df"
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
CATEGORY_COLORS = {"locked": PALETTE[2], "locked (lock check only)": PALETTE[5],
                   "lost lock during the measurement": PALETTE[1], "lost lock (lock check only)": PALETTE[4],
                   "lock bit set, check failed": PALETTE[3], "not locked": PALETTE[7], "no VCO setting": "#a3a29e"}
OFFSET_AXIS = dict(type="log", tickvals=[1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["100 Hz", "1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))
N_BINS = 200  # points per plotted phase-noise curve; None = every measured point
COLORSCALE = "Viridis"  # curves coloured by the VCO frequency (log scale)

# Measured points (carrier at the right frequency), each with a colour on COLORSCALE by log10(f_vco) over the sweep.
# Points that lost the lock during the measurement are included, drawn dotted in the curve plots.
ok = runs[runs["measured"]]
_lf = np.log10(runs["f_vco"])
COLOR_OF = dict(zip(runs["f_ref"], sample_colorscale(COLORSCALE, list((_lf - _lf.min()) / max(np.ptp(_lf), 1e-9)))))


def dash_of(r):
    return "solid" if r.good else "dot"


def lost_txt(r):
    return "" if r.good else " (lost lock during the measurement)"


def trace_of(f_ref, kind):
    return traces[(traces["f_ref"] == f_ref) & (traces["kind"] == kind)]


def in_range(df, f_start, f_stop):
    """Rows of df with the offset (column freq) between f_start and f_stop, so the y axis fits the shown band."""
    return df[(df["freq"] >= f_start) & (df["freq"] <= f_stop)]


def binned(df, f_start, f_stop):
    """The curve in df (columns freq, level_dbm) between f_start and f_stop, averaged into N_BINS log-spaced bins in
    linear power (pll_util.bin_phase_noise): display only, the jitter uses the measured points."""
    df = in_range(df, f_start, f_stop).sort_values("freq")
    if N_BINS is None or len(df) < 2:
        return df
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], np.geomspace(f_start, f_stop, N_BINS + 1))
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def add_colorbar(fig):
    """Colour bar for the VCO frequency (invisible trace carrying the colour scale)."""
    fig.add_scatter(x=[None], y=[None], mode="markers", showlegend=False, hoverinfo="skip",
                    marker=dict(colorscale=COLORSCALE, cmin=_lf.min(), cmax=_lf.max(), color=[_lf.min()],
                                colorbar=dict(title="VCO", tickvals=np.log10([2e8, 5e8, 1e9, 2e9, 5e9, 1e10]),
                                              ticktext=["200 MHz", "500 MHz", "1 GHz", "2 GHz", "5 GHz", "10 GHz"])))


def save(fig, name, csv=None):
    """Interactive HTML and a static PNG (kaleido) next to `csv` (None: RESULT_CSV), then show the figure."""
    csv = Path(csv) if csv is not None else RESULT_CSV
    base = csv.with_name(f"{csv.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [19]:
# Lock range overview against the reference frequency (VCO = 128 x ref): result per point and, when it changes over
# the sweep, the loop filter of every point (top), PLL supply power (every configured point), RMS jitter (measured
# points: full curve total, noise only, spurs) and the lookup table's expected Vctrl of the chosen VCO setting.
# Shaded: the lock range (longest stretch of locked points).
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.13, 0.29, 0.29, 0.29])
hover = "ref %{x:.3f} MHz (VCO %{customdata:.0f} MHz), filter %{hovertext}"
for cat, color in CATEGORY_COLORS.items():
    d = runs[runs["category"] == cat]
    if d.empty:
        continue
    fig.add_scatter(x=d["f_ref"] / 1e6, y=[cat] * len(d), mode="markers", name=cat, row=1, col=1,
                    marker=dict(color=color, size=10, symbol="square"), customdata=d["f_vco"] / 1e6,
                    hovertext=d["filter"], hovertemplate=f"{cat}<br>{hover}<extra></extra>")
    fig.add_scatter(x=d["f_ref"] / 1e6, y=d["pll_p"] * 1e3, mode="markers", showlegend=False, row=2, col=1,
                    marker=dict(color=color, size=8), customdata=d["f_vco"] / 1e6, hovertext=d["filter"],
                    hovertemplate=f"%{{y:.3f}} mW<br>{hover}<extra></extra>")
    fig.add_scatter(x=d["f_ref"] / 1e6, y=d["vctrl_pred"], mode="markers", showlegend=False, row=4, col=1,
                    marker=dict(color=color, size=8), customdata=d["f_vco"] / 1e6, hovertext=d["filter"],
                    hovertemplate=f"Vctrl %{{y:.3f}} V<br>{hover}<extra></extra>")
if FILTER_TXT == "per point":  # which filter covers which band
    fig.add_scatter(x=runs["f_ref"] / 1e6, y=runs["filter"], mode="markers", showlegend=False, row=1, col=1,
                    marker=dict(color=INK, size=8, symbol="line-ns-open", line=dict(width=2)),
                    customdata=runs["f_vco"] / 1e6, hovertext=runs["filter"],
                    hovertemplate=f"{hover}<extra></extra>")
for col, name, color, dash in (("time_jitter_trace_ps", "jitter: full curve, total", PALETTE[0], "solid"),
                               ("time_jitter_noise_ps", "jitter: noise only", PALETTE[2], "solid"),
                               ("time_jitter_spur_ps", "jitter: spurs", PALETTE[3], "dot")):
    fig.add_scatter(x=ok["f_ref"] / 1e6, y=ok[col], mode="lines+markers", name=name, row=3, col=1,
                    line=dict(color=color, width=2, dash=dash), marker=dict(size=7), customdata=ok["f_vco"] / 1e6,
                    hovertext=ok["filter"], hovertemplate=f"{name} %{{y:.2f}} ps<br>{hover}<extra></extra>")
if LOCK:
    fig.add_vrect(x0=LOCK[0] / 1e6, x1=LOCK[1] / 1e6, fillcolor=PALETTE[2], opacity=0.08, line_width=0)
fig.update_layout(title=f"Lock range, loop filter {FILTER_TXT}<br><sub>{RESULT_CSV.name}</sub>",
                  template="plotly_white", height=900 + (100 if FILTER_TXT == "per point" else 0),
                  legend=dict(orientation="h", y=-0.08), hoverlabel=dict(namelength=-1))
fig.update_xaxes(type="log", gridcolor=GRID)
fig.update_xaxes(title="reference frequency [MHz] (VCO = 128 x ref)", row=4)
fig.update_yaxes(gridcolor=GRID)
fig.update_yaxes(showticklabels=FILTER_TXT == "per point", row=1)  # the filter labels; the results in the legend
fig.update_yaxes(title="power [mW]", rangemode="tozero", row=2)
fig.update_yaxes(title="RMS jitter [ps]", type="log", row=3)
fig.update_yaxes(title="Vctrl pred [V]", row=4)
save(fig, "overview")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_coarse_S5_20261103_174312_overview.html and .png


In [20]:
# Comparison with COMPARE_CSV (e.g. the same filter on 2026-10-13, before the PLL supply moved to the E36312A),
# against the VCO frequency: the result per point of both runs and the RMS jitter over the common band (full curve,
# measured points; open markers: lost the lock during the measurement). The table pairs every point of this run with
# the nearest point of the other run (within MATCH_TOL in VCO frequency).
MATCH_TOL = 0.03  # relative VCO frequency difference still counted as the same point (12 GHz is 5 % off 11.4 GHz)

if COMPARE_CSV is not None:
    cmp_runs = load_runs(COMPARE_CSV)
    if filter_txt(cmp_runs) != FILTER_TXT:
        print(f"note: different loop filters: {RESULT_CSV.name} {FILTER_TXT}, "
              f"{Path(COMPARE_CSV).name} {filter_txt(cmp_runs)}")
    both = [(RESULT_CSV.stem, runs, PALETTE[0]), (Path(COMPARE_CSV).stem, cmp_runs, "#a3a29e")]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.3, 0.7])
    for name, df, color in both:
        for cat, cat_color in CATEGORY_COLORS.items():
            d = df[df["category"] == cat]
            if d.empty:
                continue
            fig.add_scatter(x=d["f_vco"] / 1e9, y=[name] * len(d), mode="markers", row=1, col=1, showlegend=False,
                            marker=dict(color=cat_color, size=10, symbol="square"),
                            hovertemplate=f"{name}: {cat}<br>VCO %{{x:.3f}} GHz<extra></extra>")
        d = df[df["measured"]]
        fig.add_scatter(x=d["f_vco"] / 1e9, y=d["time_jitter_trace_common_ps"], mode="lines+markers", name=name,
                        row=2, col=1, line=dict(color=color, width=2),
                        marker=dict(size=8, symbol=["circle" if g else "circle-open" for g in d["good"]]),
                        hovertemplate=f"{name}<br>VCO %{{x:.3f}} GHz: %{{y:.2f}} ps<extra></extra>")
    for cat, cat_color in CATEGORY_COLORS.items():  # legend of the result colours
        fig.add_scatter(x=[None], y=[None], mode="markers", name=cat, row=1, col=1,
                        marker=dict(color=cat_color, size=10, symbol="square"))
    cb = runs["common_band"].dropna()
    fig.update_layout(title=f"{RESULT_CSV.name} against {Path(COMPARE_CSV).name}, loop filter {FILTER_TXT}",
                      template="plotly_white", height=650, legend=dict(orientation="h", y=-0.12),
                      hoverlabel=dict(namelength=-1))
    fig.update_xaxes(type="log", gridcolor=GRID)
    fig.update_xaxes(title="VCO frequency [GHz]", row=2)
    fig.update_yaxes(gridcolor=GRID)
    fig.update_yaxes(title="RMS jitter, common band [ps]" + (f"<br>{cb.iloc[0]}" if len(cb) else ""), type="log",
                     row=2)
    save(fig, "compare")

    # Nearest point of the other run per point of this one
    rows = []
    for r in runs.itertuples():
        i = (np.log(cmp_runs["f_vco"]) - np.log(r.f_vco)).abs().idxmin()
        c = cmp_runs.loc[i]
        near = abs(c.f_vco / r.f_vco - 1) <= MATCH_TOL
        rows.append({"VCO [MHz]": r.f_vco / 1e6, "result": r.category,
                     "jitter [ps]": r.time_jitter_trace_common_ps, "P [mW]": r.pll_p * 1e3,
                     "other VCO [MHz]": c.f_vco / 1e6 if near else np.nan,
                     "other result": c.category if near else "-",
                     "other jitter [ps]": c.time_jitter_trace_common_ps if near else np.nan})
    display(pd.DataFrame(rows).round(2))

In [21]:
F_start, F_stop = 1e3, 10e6  # offset range shown [Hz]

# Phase noise referred to the VCO (full curve on the pad + 20 log10 N) of every measured point, coloured by the VCO
# frequency; dotted: lost the lock during the measurement. Binned to N_BINS points (linear power average, display only).
fig = go.Figure()
for r in ok.itertuples():
    d = binned(trace_of(r.f_ref, "L_trace"), F_start, F_stop)
    fig.add_scatter(x=d["freq"], y=d["level_dbm"] + 20 * np.log10(r.total_div), mode="lines", showlegend=False,
                    line=dict(color=COLOR_OF[r.f_ref], width=1.5, dash=dash_of(r)),
                    hovertemplate=f"VCO {r.f_vco / 1e6:.0f} MHz (ref {r.f_ref / 1e6:.3f} MHz), filter {r.filter}"
                                  f"{lost_txt(r)}<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>")
add_colorbar(fig)
fig.update_layout(title=f"Phase noise referred to the VCO, measured points, loop filter {FILTER_TXT}",
                  template="plotly_white", height=550)
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **OFFSET_AXIS,
                 range=[np.log10(F_start), np.log10(F_stop)])
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise")

Wait expired, Browser is being closed by watchdog.


saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_coarse_S5_20261103_174312_phase_noise.html and .png


In [22]:
F_start, F_stop = None, None  # offset from the carrier shown [Hz] (may be negative); None = edge of the traces

# Averaged spectrum on the pad around the carrier of every measured point, relative to the carrier (trace maximum), in
# the trace's RBW (not binned), coloured by the VCO frequency; dotted: lost the lock during the measurement.
fig = go.Figure()
for r in ok.itertuples():
    d = trace_of(r.f_ref, "spectrum")
    if d.empty:
        continue
    d = d.assign(freq=d["freq"] - d["center"], level_dbc=d["level_dbm"] - d["level_dbm"].max())
    d = in_range(d, d["freq"].min() if F_start is None else F_start, d["freq"].max() if F_stop is None else F_stop)
    fig.add_scatter(x=d["freq"], y=d["level_dbc"], mode="lines", showlegend=False,
                    line=dict(color=COLOR_OF[r.f_ref], width=1, dash=dash_of(r)),
                    hovertemplate=f"VCO {r.f_vco / 1e6:.0f} MHz, pad {r.f_out_meas / 1e6:.3f} MHz, "
                                  f"carrier {r.carrier_dbm:.1f} dBm{lost_txt(r)}<br>"
                                  "%{x:.4s}Hz: %{y:.1f} dBc<extra></extra>")
add_colorbar(fig)
rbw = traces.loc[traces["kind"] == "spectrum", "rbw"]
fig.update_layout(title=f"Spectrum on the pad around the carrier, measured points"
                        + (f" (RBW {rbw.iloc[0]:g} Hz)" if len(rbw) else "") + f", loop filter {FILTER_TXT}",
                  template="plotly_white", height=550)
fig.update_xaxes(title="offset from the carrier", type="linear", ticksuffix="Hz", exponentformat="SI", gridcolor=GRID)
fig.update_yaxes(title="level relative to the carrier [dBc in RBW]", gridcolor=GRID)
save(fig, "spectrum")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_coarse_S5_20261103_174312_spectrum.html and .png


In [23]:
# Power efficiency and relative jitter against the VCO frequency:
#   - PLL supply power per GHz of VCO frequency [mW/GHz], every configured point with a supply reading, coloured by
#     the result (the power is drawn also without lock);
#   - RMS jitter as a percentage of the clock period 1 / CYCLE_COL (the time jitter is the same on the VCO and on the
#     divided outputs), full curve and common band, measured points (open markers: lost the lock during the
#     measurement); COMPARE_CSV in grey when set.
CYCLE_COL = "f_vco"  # clock whose period the jitter is compared with: "f_vco", "f_pll_out" (after set_div_freq), "f_out"

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06)
for cat, color in CATEGORY_COLORS.items():
    d = runs[(runs["category"] == cat) & runs["pll_p"].notna()]
    if d.empty:
        continue
    fig.add_scatter(x=d["f_vco"] / 1e9, y=d["pll_p"] * 1e3 / (d["f_vco"] / 1e9), mode="markers", name=cat,
                    row=1, col=1, marker=dict(color=color, size=9), customdata=d["pll_p"] * 1e3,
                    hovertemplate=f"{cat}<br>VCO %{{x:.3f}} GHz: %{{y:.3f}} mW/GHz (%{{customdata:.3f}} mW)"
                                  "<extra></extra>")


def jitter_pct(df, col):
    return df[col] * 1e-12 * df[CYCLE_COL] * 100


series = [(RESULT_CSV.stem, ok, PALETTE[0], PALETTE[2])]
if COMPARE_CSV is not None:
    _c = load_runs(COMPARE_CSV)
    series.append((Path(COMPARE_CSV).stem, _c[_c["measured"]], "#a3a29e", "#cfceca"))
for name, d, c_curve, c_common in series:
    for col, label, color, dash in (("time_jitter_trace_ps", "full curve", c_curve, "solid"),
                                    ("time_jitter_trace_common_ps", "common band", c_common, "dot")):
        fig.add_scatter(x=d["f_vco"] / 1e9, y=jitter_pct(d, col), mode="lines+markers", name=f"{name}: {label}",
                        row=2, col=1, line=dict(color=color, width=2, dash=dash),
                        marker=dict(size=8, symbol=["circle" if g else "circle-open" for g in d["good"]]),
                        customdata=d[col],
                        hovertemplate=f"{name}, {label}<br>VCO %{{x:.3f}} GHz: %{{y:.2f}} % "
                                      "(%{customdata:.2f} ps)<extra></extra>")
fig.update_layout(title=f"Power per GHz and jitter relative to the clock period, loop filter {FILTER_TXT}"
                        f"<br><sub>{RESULT_CSV.name}</sub>",
                  template="plotly_white", height=750, legend=dict(orientation="h", y=-0.12),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(type="log", gridcolor=GRID)
fig.update_xaxes(title="VCO frequency [GHz]", row=2)
fig.update_yaxes(gridcolor=GRID)
fig.update_yaxes(title="PLL power / f_vco [mW/GHz]", rangemode="tozero", row=1)
fig.update_yaxes(title=f"RMS jitter [% of 1 / {CYCLE_COL}]", type="log", row=2)
save(fig, "efficiency")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_coarse_S5_20261103_174312_efficiency.html and .png


In [24]:
F_start, F_stop = None, None  # offset from the carrier shown [Hz] (may be negative); None = edge of the traces

# Wider spectrum on the pad around the carrier of every measured point: the widest sweep of the phase-noise
# measurement (kind "phase_noise", 42 MHz span = +-21 MHz, RBW 50 kHz in the runs of 2026-11-03; not averaged like
# "spectrum"), relative to the carrier (trace maximum), coloured by the VCO frequency; dotted: lost the lock during
# the measurement. Shows the reference spurs (+-f_ref / N on the pad) and anything further out.
fig = go.Figure()
spans = []
for r in ok.itertuples():
    d = trace_of(r.f_ref, "phase_noise")
    if d.empty:
        continue
    d = d[d["span"] == d["span"].max()]
    spans.append((d["span"].iloc[0], d["rbw"].iloc[0]))
    d = d.assign(freq=d["freq"] - d["center"], level_dbc=d["level_dbm"] - d["level_dbm"].max())
    d = in_range(d, d["freq"].min() if F_start is None else F_start, d["freq"].max() if F_stop is None else F_stop)
    fig.add_scatter(x=d["freq"], y=d["level_dbc"], mode="lines", showlegend=False,
                    line=dict(color=COLOR_OF[r.f_ref], width=1, dash=dash_of(r)),
                    hovertemplate=f"VCO {r.f_vco / 1e6:.0f} MHz, pad {r.f_out_meas / 1e6:.3f} MHz, "
                                  f"carrier {r.carrier_dbm:.1f} dBm{lost_txt(r)}<br>"
                                  "%{x:.4s}Hz: %{y:.1f} dBc<extra></extra>")
add_colorbar(fig)
fig.update_layout(title=f"Wide spectrum on the pad around the carrier, measured points"
                        + (f" (span {spans[0][0] / 1e6:g} MHz, RBW {spans[0][1] / 1e3:g} kHz)" if spans else "")
                        + f", loop filter {FILTER_TXT}",
                  template="plotly_white", height=550)
fig.update_xaxes(title="offset from the carrier", type="linear", ticksuffix="Hz", exponentformat="SI", gridcolor=GRID)
fig.update_yaxes(title="level relative to the carrier [dBc in RBW]", gridcolor=GRID)
save(fig, "spectrum_wide")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_coarse_S5_20261103_174312_spectrum_wide.html and .png


In [25]:
# Edges of the lock range: lock checks only (sw/tests/pll_range.py floor_search / upper_search, 2026-11-03), each
# point configured, lock bit read, read again after the hold (5 s floor, 15 s upper end):
#   - upper end: VCO 11.4-12.5 GHz on the fastest VCO setting, per PLL supply (0.75 / 0.80 / 0.85 V, measured
#     voltage rounded to 10 mV) and loop filter; the highest VCO that held the lock per supply and filter, and the
#     PLL supply power of the locked points;
#   - floor: VCO 50-200 MHz per |Kvco| request and loop filter (the weak loops); the lowest VCO that held the lock.
UPPER_CSV = None  # None = newest pll_range_upper_*.csv
FLOOR_CSV = None  # None = newest pll_range_floor_*.csv


def newest(pattern):
    paths = sorted((p for p in PLL_DIR.glob(pattern) if not p.stem.endswith("_traces")),
                   key=lambda p: p.stat().st_mtime)
    return paths[-1] if paths else None


def kvco_txt(v):
    try:
        return f"{float(v) / 1e9:g} GHz/V"
    except ValueError:
        return str(v)  # "min", "max", "middle"


def add_results(fig, d, x, y, row, shown, hover):
    """Markers of the points of d at (x, y) in subplot `row`, coloured by result; legend entry once per result."""
    for cat, color in CATEGORY_COLORS.items():
        c = d[d["category"] == cat]
        if c.empty:
            continue
        fig.add_scatter(x=x(c), y=y(c), mode="markers", name=cat, legendgroup=cat, showlegend=cat not in shown,
                        row=row, col=1, marker=dict(color=color, size=13, symbol="square"),
                        customdata=np.c_[c["pll_i"] * 1e3, c["vctrl_pred"]],
                        hovertemplate=f"{cat}<br>{hover}<br>I %{{customdata[0]:.2f}} mA, Vctrl pred "
                                      "%{customdata[1]:.3f} V<extra></extra>")
        shown.add(cat)


# Upper end against the PLL supply
upper_csv = Path(UPPER_CSV) if UPPER_CSV else newest("pll_range_upper_*.csv")
if upper_csv is not None:
    up = load_runs(upper_csv)
    up["vdd"] = up["pll_vdd"].round(2)
    filters = list(dict.fromkeys(up["filter"]))  # in the order of the run
    fig = make_subplots(rows=len(filters), cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=[f"loop filter {f}" for f in filters])
    shown = set()
    for k, flt in enumerate(filters, 1):
        add_results(fig, up[up["filter"] == flt], lambda c: c["f_vco"] / 1e9, lambda c: c["vdd"], k, shown,
                    "VCO %{x:.3f} GHz, supply %{y:.2f} V")
    fig.update_layout(title=f"Upper end of the lock range against the PLL supply (lock check, "
                            f"{up['vco_pick'].iloc[0]} VCO setting)<br><sub>{upper_csv.name}</sub>",
                      template="plotly_white", height=220 * len(filters) + 150, legend=dict(orientation="h", y=-0.08),
                      hoverlabel=dict(namelength=-1))
    fig.update_xaxes(gridcolor=GRID)
    fig.update_xaxes(title="VCO frequency [GHz]", row=len(filters))
    fig.update_yaxes(title="supply [V]", tickvals=sorted(up["vdd"].dropna().unique()), gridcolor=GRID)
    save(fig, "lock_map", upper_csv)

    top = up[up["good"]].groupby(["vdd", "filter"])["f_vco"].max().unstack("filter") / 1e9
    print("Highest VCO [GHz] that held the lock, per supply [V] and loop filter:")
    display(top.round(3))

    # PLL supply power of the locked points against the VCO frequency, per supply (mean over the filters)
    p = up[up["good"]].groupby(["vdd", "f_vco"])["pll_p"].mean().reset_index()
    fig = go.Figure()
    for j, (vdd, d) in enumerate(p.groupby("vdd")):
        fig.add_scatter(x=d["f_vco"] / 1e9, y=d["pll_p"] * 1e3, mode="lines+markers", name=f"{vdd:.2f} V",
                        line=dict(color=PALETTE[j % len(PALETTE)], width=2), marker=dict(size=8),
                        hovertemplate=f"{vdd:.2f} V<br>VCO %{{x:.3f}} GHz: %{{y:.3f}} mW<extra></extra>")
    fig.update_layout(title=f"PLL supply power at the upper end, locked points<br><sub>{upper_csv.name}</sub>",
                      template="plotly_white", height=450, legend=dict(title="PLL supply"))
    fig.update_xaxes(title="VCO frequency [GHz]", gridcolor=GRID)
    fig.update_yaxes(title="PLL power [mW]", gridcolor=GRID)
    save(fig, "power", upper_csv)

# Floor against the loop filter, per |Kvco| request (points without a table setting: "(not configured)")
floor_csv = Path(FLOOR_CSV) if FLOOR_CSV else newest("pll_range_floor_*.csv")
if floor_csv is not None:
    fl = load_runs(floor_csv)
    fl["kvco_txt"] = fl["kvco_request"].map(kvco_txt)
    kvcos = list(dict.fromkeys(fl["kvco_txt"]))
    fig = make_subplots(rows=len(kvcos), cols=1, shared_xaxes=True, vertical_spacing=0.1,
                        subplot_titles=[f"|Kvco| request {kv}" for kv in kvcos])
    shown = set()
    for k, kv in enumerate(kvcos, 1):
        add_results(fig, fl[fl["kvco_txt"] == kv], lambda c: c["f_vco"] / 1e6,
                    lambda c: np.where(c["status"] == "no_vco_setting", "(not configured)", c["filter"]), k, shown,
                    "VCO %{x:.0f} MHz, %{y}")
    fig.update_layout(title=f"Floor of the lock range against the loop filter (lock check)<br><sub>{floor_csv.name}"
                            "</sub>", template="plotly_white", height=280 * len(kvcos) + 150,
                      legend=dict(orientation="h", y=-0.1), hoverlabel=dict(namelength=-1))
    fig.update_xaxes(type="log", gridcolor=GRID)
    fig.update_xaxes(title="VCO frequency [MHz]", row=len(kvcos))
    fig.update_yaxes(gridcolor=GRID)
    save(fig, "lock_map", floor_csv)

    low = fl[fl["good"]].groupby(["kvco_txt", "filter"])["f_vco"].min().unstack("filter") / 1e6
    print("Lowest VCO [MHz] that held the lock, per |Kvco| request and loop filter:")
    display(low.round(0))

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_upper_S5_20261103_170454_lock_map.html and .png


Highest VCO [GHz] that held the lock, per supply [V] and loop filter:


filter,Icp 3 C1 3 C2 3 R1 3,Icp 7 C1 0 C2 0 R1 7,Icp 7 C1 3 C2 3 R1 3,Icp 7 C1 3 C2 3 R1 7
vdd,,,,
0.75,11.40,11.70,11.70,11.70
0.80,12.00,12.00,12.00,12.00
0.85,12.25,12.25,12.25,12.25


saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_upper_S5_20261103_170454_power.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_range_floor_S5_20261103_165653_lock_map.html and .png


Lowest VCO [MHz] that held the lock, per |Kvco| request and loop filter:


filter,Icp 0 C1 3 C2 3 R1 3,Icp 0 C1 7 C2 3 R1 1,Icp 0 C1 7 C2 3 R1 3,Icp 0 C1 7 C2 7 R1 3
kvco_txt,,,,
2 GHz/V,200.0,160.0,160.0,160.0
min,160.0,160.0,128.0,128.0
